# KeralaGAN: train on Colab

1. **Runtime → Change runtime type → T4 GPU.**
2. Upload your dataset zip to Google Drive as `MyDrive/KeralaGAN/datasets/<name>.zip`
   (inside it: `trainA/ trainB/ testA/ testB/`, made with `scripts/prepare_dataset.py`).
3. Run the cells in order. Checkpoints go to Drive, so a disconnect loses nothing:
   just run everything again and training resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Drive and code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/KeralaGAN'          # checkpoints, samples, logs, weights live here
REPO = 'https://github.com/abhinavrenjithg-sys/KeralaGAN.git'   # <- change to your repo
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
%cd /content
!test -d KeralaGAN || git clone -q {REPO}
%cd /content/KeralaGAN
!pip install -q pytorch-fid

## 2. Dataset
Set `NAME` to `monsoon`, `night` or `mural`. Set `NAME = 'toy'` to test the whole pipeline with a synthetic dataset first.

In [ ]:
NAME = 'monsoon'

if NAME == 'toy':
    !python scripts/make_toy_dataset.py --out data/toy --n 200
else:
    !unzip -q -o {OUT}/datasets/{NAME}.zip -d data/{NAME}
for split in ['trainA', 'trainB', 'testA', 'testB']:
    path = f'data/{NAME}/{split}'
    print(split, len(os.listdir(path)) if os.path.isdir(path) else 'MISSING')

## 3. Train
About 100 epochs at 256 px. With ~1,000 images per side expect roughly 3–4 minutes per epoch on a T4 with `--amp`. Re-run this cell after a disconnect: `--resume` picks up from `latest.pt`.

In [ ]:
EPOCHS = 100
!python -m keralagan.train --data data/{NAME} --out {OUT} --epochs {EPOCHS} --amp --resume --workers 2

## 4. Look at progress
Each grid row: real A, generated B, real B, generated A.

In [ ]:
import glob
from IPython.display import Image, display
samples = sorted(glob.glob(f'{OUT}/samples/{NAME}/*.png'))
print(len(samples), 'sample grids')
for p in samples[-1:]:
    print(p); display(Image(p))

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f'{OUT}/logs/{NAME}.csv')
log['t'] = log['epoch'] + log['step'] / max(1, log['step'].max() + 50)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
log.plot(x='t', y=['loss_gan', 'loss_D'], ax=ax[0], title='Adversarial losses')
log.plot(x='t', y=['loss_cyc', 'loss_idt'], ax=ax[1], title='Cycle and identity losses')
plt.show()

## 5. Evaluate (FID) and make a before/after grid

In [ ]:
!python scripts/evaluate_fid.py --data data/{NAME} --ckpt {OUT}/checkpoints/{NAME}/latest.pt --grid {OUT}/results/{NAME}_grid.png
display(Image(f'{OUT}/results/{NAME}_grid.png'))

## 6. Export the generator for the demo app (~22 MB)

In [ ]:
!python scripts/export_generator.py --ckpt {OUT}/checkpoints/{NAME}/latest.pt --out {OUT}/weights/{NAME}_G_AB.pt

## 7. Try your own photo

In [ ]:
from google.colab import files
up = files.upload()
for fn in up:
    !python -m keralagan.translate --ckpt {OUT}/weights/{NAME}_G_AB.pt --input "{fn}" --out results --side_by_side
    display(Image('results/' + os.path.splitext(fn)[0] + '.png'))